In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [11]:
seedy = 666
df = pd.read_csv('data/chembl_valid_10to30atoms.csv')

samp_size = 50

df_samp = df.sample(samp_size, random_state=seedy)
df_samp['len'] = df_samp.smiles.apply(lambda x: len(x))
df_samp = df_samp.sort_values(by='len',ignore_index=True)

In [12]:
def get_augset_from_src(k,v,aug_iter):
        
    
    src_idx = v[0]
    anc_idx = v[1]
    smi = v[4]

    augs = get_ext_augs(smi)
    augs = list(set(augs))
    
    augset = []
    if augs:
        for aug_smi in augs:
            tup = [anc_idx, src_idx, aug_iter, aug_smi]
            if tup not in augset:
                augset.append(tup)
    return augset

In [7]:
import numpy as np
from tqdm.notebook import tqdm

from extended_augs_utils import get_ext_augs
from joblib import Parallel, delayed

iters = 5
max_augs = 10

for aug_iter in tqdm(range(iters+1), total=(iters+1)):
    
    curr_augset = []
    
    if aug_iter==0:
        for row in df_samp.itertuples():
            idx = row.Index
            anc_idx = idx
            src_idx = idx
            smi = row.smiles
            tup = [idx, anc_idx, src_idx, aug_iter, smi]
            curr_augset.append(tup)
            df_augset = pd.DataFrame(curr_augset, columns=['idx','anc_idx','src_idx', 'aug_iter', 'smiles'])
        
    else:
        df_src = df_augset[df_augset.aug_iter==(aug_iter-1)]
        
#         df_src = list(df_src.rename(columns=str.lower).itertuples())
#         df_src = [tuple(x) for x in df_src]
        dict_src = df_src.T.to_dict('list')
#         print(dict_src)
        
#         for k,v in dict_src.items():
#             print(k,v)
            
        parallelizer = Parallel(n_jobs=-1, backend='multiprocessing' )
        tasks = (delayed(get_augset_from_src)(k,v,aug_iter) for k,v in dict_src.items())
        curr_augset = parallelizer(tasks)
        
#         print([x for x in curr_augset])
#         poo = [x[1][0].__dict__ for x in curr_augset]        
#         print([type(x[1][0]) for x in curr_augset][0])
#         print(poo)
        curr_augset = [x for l in curr_augset for x in l]
        
        
    
    
    
#         flat = [x for x in flat]
#         flat = [x for l in flat for x in l]
#         print(flat)
        
#         poo = [x[1][0].__dict__ for x in curr_augset]
#         print(poo)
        curr_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        curr_augset.drop_duplicates(keep='first', inplace=True)
#         display(curr_augset)
        
        trunc_idc = []
        for i in range(len(df_samp)):
            df_anc = curr_augset[curr_augset.anc_idx==i]
            df_anc = df_anc.sample(n=10, replace=True)
            idc = df_anc.index.values
            trunc_idc.extend(idc)
        trunc_augset = curr_augset.iloc[trunc_idc]
        
        df_augset = pd.concat([df_augset, trunc_augset], axis=0)
        
        display(df_augset)

  0%|          | 0/6 [00:00<?, ?it/s]

,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
9168,999,999,1,O=C1c2cccc3cccc(c23)C(=O)N1c1cc(S)c([N+](=O)[O...
9160,999,999,1,Cc1ccc2cccc3c2c1C(=O)N(c1ccc([N+](=O)[O-])cc1[...
9166,999,999,1,Cc1c([N+](=O)[O-])ccc(N2C(=O)c3cccc4cccc(c34)C...
9161,999,999,1,O=C1c2cccc3cccc(c23)C(=O)N1c1c(O)cc([N+](=O)[O...


,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
58623,999,9161,2,Cc1cc2c3c(cccc3c1)C(=O)N(c1c(O)cc([N+](=O)[O-]...
58640,999,9166,2,Cc1c([N+](=O)[O-])c(O)cc(N2C(=O)c3cccc4cccc(c3...
58601,999,9163,2,Cc1cc(N2C(=O)c3cccc4cc(O)cc(c34)C2=O)c([N+](=O...
58624,999,9161,2,Cc1c([N+](=O)[O-])cc([N+](=O)[O-])c(N2C(=O)c3c...


,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
87535,999,58602,3,CCc1cc2c3c(cc(O)cc3c1)C(=O)N(c1ccc([N+](=O)[O-...
87557,999,58607,3,Cc1cc([N+](=O)[O-])cc([N+](=O)[O-])c1N1C(=O)c2...
87582,999,58624,3,Cc1cc2c3c(cccc3c1)C(=O)N(c1c([N+](=O)[O-])cc([...
87569,999,58623,3,Cc1cc2c3c(c(C)ccc3c1)C(=O)N(c1c(O)cc([N+](=O)[...


,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
91425,999,87549,4,Cc1cc2c3c(c(C)c(O)c(O)c3c1)C(=O)N(c1ccc([N+](=...
91431,999,87586,4,Cc1ccc2c(C)ccc3c2c1C(=O)N(c1c([N+](=O)[O-])cc(...
91465,999,87557,4,Cc1c([N+](=O)[O-])cc([N+](=O)[O-])c(N2C(=O)c3c...
91433,999,87586,4,Cc1cc2c3c(c(C)ccc3c1)C(=O)N(c1c([N+](=O)[O-])c...


,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
92438,999,91433,5,CCc1ccc2cc(C)cc3c2c1C(=O)N(c1c([N+](=O)[O-])cc...
92360,999,91451,5,CCc1cc2c3c(cc(O)c(O)c3c1)C(=O)N(c1cc(C)c([N+](...
92383,999,91473,5,Cc1cc2c3c(c(C)ccc3c1)C(=O)N(c1c(O)c(C)c([N+](=...
92431,999,91433,5,Cc1cc2ccc(C)c3c2c(c1C)C(=O)N(c1c([N+](=O)[O-])...


In [8]:
df_augset

,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
92438,999,91433,5,CCc1ccc2cc(C)cc3c2c1C(=O)N(c1c([N+](=O)[O-])cc...
92360,999,91451,5,CCc1cc2c3c(cc(O)c(O)c3c1)C(=O)N(c1cc(C)c([N+](...
92383,999,91473,5,Cc1cc2c3c(c(C)ccc3c1)C(=O)N(c1c(O)c(C)c([N+](=...
92431,999,91433,5,Cc1cc2ccc(C)c3c2c(c1C)C(=O)N(c1c([N+](=O)[O-])...


In [9]:
df_augset_nodups = df_augset.drop_duplicates(keep='first')
df_augset_nodups

,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...
92438,999,91433,5,CCc1ccc2cc(C)cc3c2c1C(=O)N(c1c([N+](=O)[O-])cc...
92360,999,91451,5,CCc1cc2c3c(cc(O)c(O)c3c1)C(=O)N(c1cc(C)c([N+](...
92383,999,91473,5,Cc1cc2c3c(c(C)ccc3c1)C(=O)N(c1c(O)c(C)c([N+](=...
92431,999,91433,5,Cc1cc2ccc(C)c3c2c(c1C)C(=O)N(c1c([N+](=O)[O-])...


In [10]:
df_augset_nodups.to_csv('20220603_extended_augset.csv',index=False)